# NumPy Tips & Tricks
### Credit Card Risk Analysis Track

This one's a **reference notebook**, not a quiz — no "YOUR CODE HERE" cells. Each tip is a short explanation + a verified, runnable example. Skim it top to bottom, then come back to it whenever you hit one of these situations for real.

All 18 snippets below were executed end-to-end with no errors before this notebook was assembled.

## Setup

In [ ]:
import numpy as np
import warnings
np.set_printoptions(suppress=True)

### 1. `np.isin` for fast membership checks
Checking "is this ID in that list" with a Python loop is slow at scale. `np.isin` vectorizes it — handy for flagging customers on a watchlist.

In [ ]:
customer_ids = np.array([101, 102, 103, 104, 105, 106, 107, 108])
watchlist = np.array([102, 105, 108, 999])
on_watchlist = np.isin(customer_ids, watchlist)
print(on_watchlist)

### 2. `np.nonzero` / `np.argwhere` to find *where* a condition is true
Boolean masks tell you *which values* match. These tell you the *positions* — useful when you need to cross-reference against another array by index.

In [ ]:
risk_scores = np.array([720, 580, 690, 610, 750, 640, 800, 560])
low_score_positions = np.nonzero(risk_scores < 600)[0]
print(low_score_positions)

# argwhere gives the same thing, shaped as a column
low_score_positions_2 = np.argwhere(risk_scores < 600).flatten()
print(low_score_positions_2)

### 3. Views vs. copies — the gotcha that causes silent bugs
**Basic slicing** (`arr[1:3]`) returns a *view* — editing it edits the original array. **Fancy indexing** (`arr[[1,2]]`) returns a *copy* — editing it does NOT touch the original. Mixing these up is one of the most common sources of "why did my raw data change?!" bugs.

In [ ]:
balances = np.array([1200.0, 3400.0, 0.0, 8900.0])
view_slice = balances[1:3]
view_slice[0] = 99999.0
print("basic slice IS a view -> original changed:", balances)

balances2 = np.array([1200.0, 3400.0, 0.0, 8900.0])
fancy_slice = balances2[[1, 2]]
fancy_slice[0] = 99999.0
print("fancy indexing IS a copy -> original unchanged:", balances2)

### 4. In-place operators (`*=`, `+=`) to avoid extra memory allocation
`arr = arr * 1.02` allocates a brand-new array. `arr *= 1.02` modifies the existing buffer. On a dataset with millions of rows, this difference is real.

In [ ]:
big = np.arange(5, dtype=float)
big *= 1.02
print(big)

### 5. Never compare floats with `==` — use `np.allclose`
Floating-point rounding means two "equal" quantities (like a computed balance vs. an expected one) often aren't bit-identical.

In [ ]:
a = np.array([0.1 + 0.2, 1.0])
b = np.array([0.3, 1.0000000001])
print(np.array_equal(a, b))       # False -- looks wrong but isn't
print(np.allclose(a, b, atol=1e-6))  # True -- this is the check you actually want

### 6. Downcast dtypes to cut memory usage on large datasets
You rarely need `int64` for a customer ID under a billion, or `float64` for a balance. Downcasting can cut memory in half or more — matters a lot once you're loading a full transactions dataset into pandas (which uses numpy dtypes under the hood).

In [ ]:
ids64 = np.array([101, 102, 103], dtype=np.int64)
ids32 = ids64.astype(np.int32)
print(ids64.nbytes, "bytes vs", ids32.nbytes, "bytes")

### 7. `np.cumsum` for running totals
Turns daily deposits/withdrawals directly into a running account balance — no loop needed.

In [ ]:
daily_deposits = np.array([100, 250, -50, 400, -200])
running_balance = np.cumsum(daily_deposits)
print(running_balance)

### 8. `np.diff` for period-over-period change
The mirror image of `cumsum` — given a balance over time, get the change from each period to the next.

In [ ]:
monthly_balance = np.array([1200, 1350, 1300, 1500, 1450])
month_over_month_change = np.diff(monthly_balance)
print(month_over_month_change)

### 9. Broadcasting trick: pairwise differences with `[:, None]` / `[None, :]`
Reshaping one copy of an array into a column and leaving the other as a row lets broadcasting build the full pairwise comparison matrix in one line — no nested loop over every pair of customers.

In [ ]:
scores = np.array([720, 580, 690])
pairwise_diff = scores[:, None] - scores[None, :]
print(pairwise_diff)

### 10. `np.errstate` to silence expected warnings (e.g. division by zero)
When you're intentionally dividing by a value that might be zero (and handling it with `np.where`), numpy still prints a runtime warning by default. `np.errstate` scopes the suppression to just that calculation instead of silencing warnings globally.

In [ ]:
denominators = np.array([100, 0, 50, 0])
numerators = np.array([10, 20, 0, 5])
with np.errstate(divide="ignore", invalid="ignore"):
    ratios = np.where(denominators != 0, numerators / denominators, 0.0)
print(ratios)

### 11. `np.select` beats nested `np.where` once you have 3+ conditions
Nested `np.where(a, x, np.where(b, y, np.where(c, z, default)))` gets unreadable fast. `np.select` lists conditions and choices in parallel — much easier to audit for a risk-tiering rule.

In [ ]:
tiers = np.select(
    [scores >= 700, scores >= 600],
    ["Prime", "Near-Prime"],
    default="Subprime",
)
print(tiers)

### 12. `.any(axis=...)` / `.all(axis=...)` for row/column-wise checks
Answers questions like "which customers were ever late?" or "which customers were always on time?" in one line, without looping over rows.

In [ ]:
late_payments_matrix = np.array([
    [0, 0, 1],
    [0, 0, 0],
    [1, 1, 0],
])
ever_late = late_payments_matrix.any(axis=1)
always_on_time = ~late_payments_matrix.any(axis=1)
print(ever_late)
print(always_on_time)

### 13. `np.repeat` vs `np.tile` — easy to mix up
`repeat` repeats **each element** in place (`A A A B B B`). `tile` repeats **the whole array** as a block (`A B A B A B`). Useful when building test data or expanding categories to match another array's length.

In [ ]:
account_types = np.array(["checking", "savings"])
repeated = np.repeat(account_types, 3)
tiled = np.tile(account_types, 3)
print(repeated)
print(tiled)

### 14. `np.save` / `np.load` to cache expensive results
If a computation (a large simulation, a cleaned array) takes a while, save it to a `.npy` file once instead of recomputing every time you rerun the notebook.

In [ ]:
expensive_result = np.random.default_rng(1).normal(size=5)
np.save("/tmp/cached_result.npy", expensive_result)
reloaded = np.load("/tmp/cached_result.npy")
print(np.array_equal(expensive_result, reloaded))

### 15. Quick array introspection: `.shape`, `.dtype`, `.nbytes`
Before running an expensive operation on an unfamiliar array (e.g. one loaded from a big CSV), check its shape/dtype/memory footprint first — it's instant and avoids surprises.

In [ ]:
sample = np.zeros((1000, 50), dtype=np.float32)
print(sample.shape, sample.dtype, sample.nbytes, "bytes")

### 16. `out=` parameter to avoid allocating a new array
Many numpy functions accept `out=` to write results into an existing array instead of creating a new one — useful in tight loops or when memory is tight.

In [ ]:
arr1 = np.arange(5, dtype=float)
arr2 = np.empty_like(arr1)
np.multiply(arr1, 2, out=arr2)
print(arr2)

### 17. `np.nan_to_num` to clean bad values before modeling
Real financial data has missing values (`NaN`) and occasional bad divisions (`inf`). Most ML models choke on these. `nan_to_num` replaces them with sane defaults in one call.

In [ ]:
messy = np.array([1.0, np.nan, 3.0, np.inf, -np.inf])
cleaned = np.nan_to_num(messy, nan=0.0, posinf=1e6, neginf=-1e6)
print(cleaned)

### 18. `np.clip` instead of manual if/else bounds-checking
A one-liner to enforce valid ranges (e.g. credit scores must be between 300-850) — no loop, no `if val < min: val = min` boilerplate.

In [ ]:
raw_scores = np.array([-10, 250, 720, 900, 500])
valid_scores = np.clip(raw_scores, 300, 850)
print(valid_scores)

### Bonus anti-tip: `np.vectorize` is not actually fast
It's tempting to reach for `np.vectorize` to "numpy-ify" a Python function. It's convenient, but under the hood it's still calling your Python function once per element — it does **not** run in compiled C like real vectorized operations (`+`, `*`, `np.where`, etc.) do. Prefer genuinely vectorized expressions when performance matters; only reach for `np.vectorize` for readability on small arrays.

## ✅ Checkpoint

**What you covered:** 18 practical numpy idioms — fast membership/position checks, the views-vs-copies gotcha, safe float comparison, memory-conscious dtypes, running totals/diffs, broadcasting for pairwise comparisons, warning suppression scoped to one calculation, `np.select` for readable multi-tier logic, axis-wise `any`/`all`, `repeat` vs `tile`, caching with `save`/`load`, quick introspection, `out=` for memory reuse, `nan_to_num` for cleaning, `clip` for bounds, and why `np.vectorize` isn't a real performance trick.

**Why it matters for the project:** these are the idioms that show up in real, messy financial data — missing values, dtype bloat on large files, off-by-one confusion between views and copies — not just in tidy exercise data.

**What's next:** pandas — Series, DataFrames, `.loc`/`.iloc`, filtering, groupby, merging — applied directly to a credit card transactions dataset.

Let me know when you're ready to move on, or if there's a specific numpy scenario you want more tricks for.